In [0]:
from pyspark.sql import functions as F

bronze_df = spark.read.table("nyc_taxi.bronze.yellow_trips_raw")

silver_df = (
    bronze_df
    .withColumn("pickup_datetime",F.col("tpep_pickup_datetime").cast("timestamp"))
    .withColumn("dropoff_datetime",F.col("tpep_dropoff_datetime").cast("timestamp"))
    .withColumn("passenger_count",F.col("passenger_count").cast("integer"))
    .withColumn("trip_distance",F.col("trip_distance").cast("float"))
    .withColumn("fare_amount",F.col("fare_amount").cast("float"))
    .withColumn("total_amount",F.col("total_amount").cast("float"))
    .withColumn("pu_location_id",F.col("PULocationID").cast("integer"))
    .withColumn("do_location_id",F.col("DOLocationID").cast("integer"))

    .filter(
        (F.col("trip_distance") > 0) &
        (F.col("trip_distance") < 100) &
        (F.col("fare_amount") > 0) &
        (F.col("fare_amount") < 1000) &
        (F.col("passenger_count") > 0) &
        (F.col("dropoff_datetime") > F.col("pickup_datetime"))
    )

    .dropDuplicates(["pickup_datetime","dropoff_datetime","pu_location_id","do_location_id"])

    .select(
        "pickup_datetime",
        "dropoff_datetime",
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "total_amount",
        "pu_location_id",
        "do_location_id",
        "ingestion_timestamp"
    )
)

(
    silver_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable("nyc_taxi.silver.yellow_trips_clean")
)

print("Silver transformation and cleaning complete!")